## Parallel Workflow with MAF and Foundry

![parallel-workflow](./Images/parallel-workflow.png)

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.8

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, handler
from agent_framework import WorkflowBuilder, WorkflowViz

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

await client.configure_azure_monitor(
    enable_sensitive_data=True
)

### Define the Web Search Tool

In [ ]:
web_search_tool = FoundryChatClient.get_web_search_tool()

### Create the Location Picker Agent

In [ ]:
location_picker_agent = Agent(
    client=client,

    name="Location-Picker-Agent",

    instructions="""You are a helpful assistant that helps users pick a location for their vacation.""",

    tools=[
        web_search_tool
    ]
)

### Create the Destination Recommender Agent

In [ ]:
destination_recommender_agent = Agent(
    client=client,

    name="Destination-Recommender-Agent",

    instructions="""You are a travel expert that provides personalized vacation recommendations based on user preferences and locations. """,

    tools=[
        web_search_tool
    ]
)

### Create the Weather Agent

In [ ]:
weather_agent = Agent(
    client=client,

    name="Weather-Agent",

    instructions="""You are a weather expert that provides accurate and up-to-date weather information for various locations selected""",

    tools=[
        web_search_tool
    ]
)

### Create the Cuisine Suggestion Agent

In [ ]:
cuisine_suggestion_agent = Agent(
    client=client,

    name="Cuisine-Suggestion-Agent",

    instructions="""You are a culinary expert that suggests popular local cuisines and dining options based on the selected vacation destinations.""",

    tools=[
        web_search_tool
    ]
)

### Create the Itinerary Planner Agent

In [ ]:
itinerary_planner_agent = Agent(
    client=client,

    name="Itinerary-Planner-Agent",

    instructions="""You are an itinerary planning expert that creates detailed travel itineraries based on user preferences, selected destinations, weather conditions, and local cuisine options.""",

    tools=[
        web_search_tool
    ]
)

### Create the Location Selector Agent Executor

In [ ]:
class LocationSelectorExecutor(Executor):

    @handler
    async def handle(self, user_query: str, ctx: WorkflowContext[str]) -> None:
        response = await location_picker_agent.run(user_query)

        await ctx.send_message(str(response))

### Create the Destination Recommender Agent Executor

In [ ]:
class DestinationRecommenderExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await destination_recommender_agent.run(location)

        await ctx.send_message(str(response))

### Create the Weather Agent Executor

In [ ]:
class WeatherExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await weather_agent.run(location)

        await ctx.send_message(str(response))

### Create the Cuisine Suggestion Agent Executor

In [ ]:
class CuisineSuggestionExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await cuisine_suggestion_agent.run(location)

        await ctx.send_message(str(response))

### Create the Itinerary Planner Agent Executor

In [ ]:
class ItineraryPlannerExecutor(Executor):

    @handler
    async def handle(self, results: list[str], ctx: WorkflowContext[str]) -> None:
        response = await itinerary_planner_agent.run(results)

        await ctx.yield_output(str(response))

### Building the Parallel Workflow

In [ ]:
# Create the executor instances / objects
location_selector_executor = LocationSelectorExecutor(id="LocationSelector")
destination_recommender_executor = DestinationRecommenderExecutor(id="DestinationRecommender")
weather_executor = WeatherExecutor(id="Weather")
cuisine_suggestion_executor = CuisineSuggestionExecutor(id="CuisineSuggestion")
itinerary_planner_executor = ItineraryPlannerExecutor(id="ItineraryPlanner")

# Build the workflow
workflow = (
    WorkflowBuilder(start_executor=location_selector_executor)
    .add_fan_out_edges(location_selector_executor, [destination_recommender_executor, weather_executor, cuisine_suggestion_executor])
    .add_fan_in_edges([destination_recommender_executor, weather_executor, cuisine_suggestion_executor], itinerary_planner_executor)
    .build()
)

viz = WorkflowViz(workflow)

### Generate Mermaid Diagram for Visualization

In [ ]:
mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

### Run the Workflow and Stream Events

In [ ]:
async def main():

    async for event in workflow.run(
        "help me plan a vacation to India with the following details: I love historical sites, prefer warm weather, and enjoy trying local foods.",
        stream=True
    ):

        print(f"Event: {event}")

        if event.type == "output":
            print("\n=== Workflow Output ===\n")
            print(event.data)


await main()